# Análisis de sensores industriales

**Aviso:** los datos de este proyecto son **simulados**.

En este notebook hacemos paso a paso el análisis del archivo `data/sensores_industriales.csv`.
Para ejecutarlo: menú **Run → Run All Cells** (o celda por celda con `Shift + Enter`).

> Importante: este notebook debe estar en la carpeta principal del proyecto (junto a la carpeta `data`), para que las rutas relativas funcionen.

## Paso 0. Importar bibliotecas y definir rutas

`pandas` sirve para trabajar con tablas. `Path` sirve para escribir rutas que funcionan en cualquier computadora (relativas, sin `C:\...`).

In [1]:
import pandas as pd
from pathlib import Path

RUTA_CSV = Path("data") / "sensores_industriales.csv"
RUTA_ALERTAS = Path("resultados") / "alertas.csv"
UMBRAL = 85  # °C, regla didáctica del examen

## Paso 1. Leer el CSV y echar un vistazo

`df` (DataFrame) es la tabla con todos los datos. `head()` muestra las primeras 5 filas.

In [2]:
df = pd.read_csv(RUTA_CSV)
df.head()

,id_registro,fecha_hora,id_sensor,planta,temperatura_c,vibracion_mm_s
0,1,01/09/26 0:00,S001,Planta_1,65.93,1.40
1,2,01/09/26 0:00,S002,Planta_1,54.05,4.21
2,3,01/09/26 0:00,S003,Planta_1,66.34,2.26
3,4,01/09/26 0:00,S004,Planta_1,65.24,1.32
4,5,01/09/26 0:00,S005,Planta_1,68.12,3.22


## Paso 2. Cantidad de registros y de sensores distintos (1 pt)

`len(df)` cuenta las filas. `nunique()` cuenta valores diferentes en una columna.

In [3]:
print(f"Cantidad de registros: {len(df):,}")
print(f"Sensores distintos: {df['id_sensor'].nunique()}")

Cantidad de registros: 100,000
Sensores distintos: 40


## Paso 3. Temperatura promedio de cada planta (2 pts)

`groupby('planta')` agrupa las filas por planta y `mean()` calcula el promedio de cada grupo.

In [4]:
promedios = df.groupby("planta")["temperatura_c"].mean().round(2)
promedios

planta
Planta_1    66.62
Planta_2    66.53
Planta_3    66.77
Planta_4    66.67
Name: temperatura_c, dtype: float64

## Paso 4. Temperatura máxima, sensor y fecha (1 pt)

Primero buscamos el valor máximo y luego **todas** las filas que lo tengan (así, si hay empates, aparecen todos).

In [5]:
t_max = df["temperatura_c"].max()
filas_max = df[df["temperatura_c"] == t_max]

print(f"Temperatura máxima: {t_max} °C ({len(filas_max)} lectura(s) con ese valor)")
filas_max[["id_registro", "id_sensor", "planta", "fecha_hora", "temperatura_c"]]

Temperatura máxima: 104.99 °C (4 lectura(s) con ese valor)


,id_registro,id_sensor,planta,fecha_hora,temperatura_c
53742,53743,S023,Planta_3,01/09/26 22:23,104.99
89258,89259,S019,Planta_2,02/09/26 13:11,104.99
94533,94534,S014,Planta_2,02/09/26 15:23,104.99
96109,96110,S030,Planta_3,02/09/26 16:02,104.99


## Paso 5. Lecturas con temperatura mayor que 85 °C (1 pt)

Filtramos las filas con `df[df['temperatura_c'] > UMBRAL]`. Ojo: es **mayor que** (>), no mayor o igual.

In [6]:
alertas = df[df["temperatura_c"] > UMBRAL]
print(f"Total de alertas (> {UMBRAL} °C): {len(alertas):,}")

Total de alertas (> 85 °C): 6,954


## Paso 6. Planta con más alertas (1 pt)

`value_counts()` cuenta cuántas alertas hay por planta. Después buscamos todas las plantas que tengan el máximo (por si hay empate).

In [7]:
alertas_por_planta = alertas["planta"].value_counts()
print(alertas_por_planta.to_string())

maximo = alertas_por_planta.max()
ganadoras = alertas_por_planta[alertas_por_planta == maximo].index.tolist()
print()
print(f"Planta(s) con más alertas: {', '.join(ganadoras)} ({maximo:,} alertas)")

planta
Planta_3    1777
Planta_1    1737
Planta_4    1732
Planta_2    1708

Planta(s) con más alertas: Planta_3 (1,777 alertas)


## Paso 7. Exportar las alertas a `resultados/alertas.csv` (2 pts)

`index=False` evita que pandas agregue una columna extra de números, así se conservan **solo las columnas originales**.

In [8]:
RUTA_ALERTAS.parent.mkdir(exist_ok=True)   # crea la carpeta resultados/ si no existe
alertas.to_csv(RUTA_ALERTAS, index=False)
print(f"Se guardaron {len(alertas):,} alertas en {RUTA_ALERTAS}")

Se guardaron 6,954 alertas en resultados/alertas.csv


## Paso 8 (verificación). Revisar el archivo exportado

In [9]:
check = pd.read_csv(RUTA_ALERTAS)
print("Columnas:", list(check.columns))
print("Columnas iguales al original:", list(check.columns) == list(df.columns))
print("Filas:", len(check))
print("Mínima temperatura exportada:", check["temperatura_c"].min(), "(debe ser > 85)")

Columnas: ['id_registro', 'fecha_hora', 'id_sensor', 'planta', 'temperatura_c', 'vibracion_mm_s']
Columnas iguales al original: True
Filas: 6954
Mínima temperatura exportada: 85.01 (debe ser > 85)
